## **Previsão preventiva de insatisfação de clientes**
## **no e-commerce (Olist)**

#### esse projeto busca criar um modelo de machine learning capaz de prever a nota de avaliação do cliente, identificando de maneira eficiente e preventiva, possíveis detratores que possam dar uma nota baixa, impactando na reputação da plataforma. 

#### O algoritmo consegue auxiliar o time de CRM, permitindo que sejam realizadas ações preventivas de retenção, antes mesmo do pedido chegar.

### **Parte I - Integração, limpeza e análise exploratória de Dados**

### 🎯 **Objetivo desta Etapa:**

#### consolidar as tabelas relacionais clássicas do ecossistema Olist em um único ecossistema estável de dados, realizando o diagnóstico de valores nulos e duplicados e culminando em uma Análise Exploratória (EDA) focada em extrair os primeiros insights sobre o faturamento e os fatores que impactam a satisfação do cliente.

## **bibliotecas**

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import joblib

## **Etapa I - base de dados**

#### O conjunto de dados contem os pedidos realizados de um e-commerce: 'Olist Store', trazendo informações de quase 100 mil pedidos feitos entre 2016 e 2018 em diversos marketplaces no Brasil. As 7 bases permitem visualizar um pedido sob múltiplas perspectivas: desde o status do pedido, preço, pagamento e desempenho do frete até a localização do cliente, atributos do produto, a geolocalização que relaciona CEPs brasileiros a coordenadas de latitude e longitude e, por fim, avaliações escritas pelos clientes.

In [ ]:
# lendo as bases de dados
df_pedidos = pd.read_csv('../bases/bases_brutas/olist_orders_dataset.csv')
df_clientes = pd.read_csv('../bases/bases_brutas/olist_customers_dataset.csv')
df_itens = pd.read_csv('../bases/bases_brutas/olist_order_items_dataset.csv')
df_produtos = pd.read_csv('../bases/bases_brutas/olist_products_dataset.csv')
df_pagamentos = pd.read_csv('../bases/bases_brutas/olist_order_payments_dataset.csv')
df_avaliacoes = pd.read_csv('../bases/bases_brutas/olist_order_reviews_dataset.csv')
df_categorias = pd.read_csv('../bases/bases_brutas/product_category_name_translation.csv')
df_vendedores = pd.read_csv('../bases/bases_brutas/olist_sellers_dataset.csv')

In [ ]:
# criando dataframe de itens únicos para evitar duplicatas
df_itens_unicos = df_itens.groupby('order_id').agg(
    valor_produtos   = ('price', 'sum'),
    valor_frete      = ('freight_value', 'sum'),
    qtd_itens_pedido = ('order_item_id', 'count'), # Pega o número máximo de itens
    product_id       = ('product_id', 'first'), # guarda o id do produto
    seller_id        = ('seller_id', 'first'),    # Guarda o ID do vendedor para o modelo
    shipping_limit_date = ('shipping_limit_date', 'first')
).reset_index()

# juntando as formas de pagamento para consolidar o valor pago por pedido
df_pagamentos_unicos = df_pagamentos.groupby('order_id').agg(
    valor_total = ('payment_value', 'sum'),
    payment_type = ('payment_type', 'first')
).reset_index()


# agrupando avaliacoes por pedido
df_avaliacoes_unicas = df_avaliacoes.groupby('order_id')['review_score'].min().reset_index()

In [ ]:
# integrando as bases para uma melhor análise
base = (
    df_pedidos                                             
    .merge(df_clientes, on='customer_id', how='left')      
    .merge(df_itens_unicos, on='order_id', how='left')     
    .merge(df_pagamentos_unicos, on='order_id', how='left')
    .merge(df_avaliacoes_unicas, on='order_id', how='left') # Pegando a sua variável certinha
    .merge(df_produtos, on='product_id', how='left')       # <-- Agora o Pandas vai achar o product_id!
    .merge(df_vendedores, on='seller_id', how='left')      # <-- E o seller_id também!
    .merge(df_categorias, on='product_category_name', how='left') 
)


display(base.head())
print('\n==================================')
print('bases unificadas e volurimetria protegida:\n')
print(f'{base.shape[0]} linhas | {base.shape[1]} colunas')
print('==================================')

## **Etapa II - Pré processamento e limpeza**

### **Remoção de algumas colunas desnecessárias nessa etapa**

In [ ]:
colunas_para_apagar = [
    'order_approved_at',                       # data de aprovação
    'review_comment_title',                      # título do comentário
    'review_comment_message',                   # mensagem do comentário
    'review_creation_date',                     # data que escreveu o comentário
    'review_answer_timestamp',                   # resposta da avaliação
    'product_name_lenght',                      # tamanho do nome do produto
    'product_description_lenght',               # tamanho da descrição
    'product_photos_qty',                       # quantidade de fotos
    'seller_zip_code_prefix',                   # cep do vendedor
    'product_category_name_english',             # já há a coluna em português
    'customer_zip_code_prefix',                 # (cep) alta granularidade, transformado em regiao
    'order_item_id',                            # apenas identificador
]

base = base.drop(colunas_para_apagar, axis=1, errors='ignore')

## **Análise e tratamento de nulos**

#### decisões baseadas em percentual e impactos na análise.

In [ ]:
# calculando os nulos por coluna
contagem_nulos = base.isnull().sum()

# filtrando exibindo apenas as colunas que têm pelo menos 1 nulo
colunas_com_nulos = contagem_nulos[contagem_nulos > 0]

#transformando em dataframe
colunas_com_nulos=colunas_com_nulos.reset_index()

# proporção de nulos
colunas_com_nulos['percentual'] = (colunas_com_nulos[0] / len(base)).mul(100).round(2)

colunas_com_nulos=colunas_com_nulos.rename(columns={
    'index': 'coluna', 
    0: 'qtde_nulos'        
})

# ordenando
colunas_com_nulos.sort_values(by='qtde_nulos', ascending=False)

In [ ]:
# nulos na coluna peso do produto serão substituidos pela mediana da coluna
base['product_weight_g'] = base['product_weight_g'].fillna(base['product_weight_g'].median())

# nulos na coluna dimensões do produto serão substituidos pela mediana da coluna
base['product_length_cm'] = base['product_length_cm'].fillna(base['product_length_cm'].median())

# notas da avaliação nulas serão substituidos por -1 significando 'não avaliado'
base['review_score'] = base['review_score'].fillna(-1) # -1 = Não avaliado

## **exclusão de linhas nulas**

In [ ]:
base = base.dropna(subset=[
    # data entrega ao cliente é uma das colunas centrais do projeto, linhas sem essa coluna são inúteis.
    'order_delivered_customer_date',
    # categoria do produto também é importante, portanto linhas sem informações na coluna, serão excluidas.
    'product_category_name',
    # dimensões do produto e entrega pela transportadora pois é apenas 1 linha
    'product_height_cm', 'payment_type', 'order_delivered_carrier_date'])

In [ ]:
base.isnull().sum()

## **Validação**

In [ ]:
# calculando os nulos por coluna
print('=============================================')
print(f'Número de nulos no dataset pós-tratamento: \n {base.isnull().sum().sum()}')
print('=============================================')

## **Análise de duplicatas**

In [ ]:
print('=============================================')
print(f'número de linhas duplicadas: \n{base.duplicated().sum()}')
print('=============================================')

## **Engenharia de atributos iniciais**

In [ ]:
# média histórica do vendedor
base = base.sort_values(
    ['seller_id', 'order_purchase_timestamp']
)

base['media_historica_vendedor'] = (
    base
    .groupby('seller_id')['review_score']
    .transform(
        lambda x: x.shift().expanding().mean()          # Calcula a média acumulada do histórico.
    )
)

# quantidade histórica de vendas - quantas vendas anteriores esse vendedor já teve.
base['qtd_vendas_historicas'] = (
    base
    .groupby('seller_id')
    .cumcount()
)

# Desvio padrão histórico das notas do vendedor - Quão instáveis foram as avaliações anteriores desse vendedor.
base['desvio_padrao_notas_vendedor'] = (
    base
    .groupby('seller_id')['review_score']
    .transform(
        lambda x: x.shift().expanding().std()
    )
)

# para substituir nulos do desvio_padrao_notas_vendedor
std_global = base['review_score'].std()

base['desvio_padrao_notas_vendedor'] = (
    base['desvio_padrao_notas_vendedor']
    .fillna(std_global)
)

# para substituir nulos da media_historica_vendedor
media_global = base['review_score'].mean()

base['media_historica_vendedor'] = (
    base['media_historica_vendedor']
    .fillna(media_global)
)

# flag de vendedor sem histórico
base['vendedor_sem_historico'] = (
    base['qtd_vendas_historicas'] == 0
).astype(int)

In [ ]:
# Valor total do pedido = Preço do produto + Frete
base['valor_total'] = base['valor_produtos'] + base['valor_frete']

# primeiro transformando a tipagem das colunas para datas
base['order_delivered_customer_date'] = pd.to_datetime(base['order_delivered_customer_date'])
base['order_purchase_timestamp'] = pd.to_datetime(base['order_purchase_timestamp'])
base['shipping_limit_date'] = pd.to_datetime(base['shipping_limit_date'])
base['order_estimated_delivery_date'] = pd.to_datetime(base['order_estimated_delivery_date'])

# Criando a flag de controle (1 se for não avaliado, 0 se foi avaliado)
base['nao_avaliado'] = np.where(base['review_score'] == -1, 1, 0)

# Criando o ALVO puramente numérico (0 e 1) para o modelo e gráficos
# Quem é -1 (não avaliado) ganha 0 por padrão para não misturar com as notas ruins reais
base['alvo'] = np.where((base['review_score'] < 4) & (base['review_score'] != -1), 1, 0)

# Tempo de entrega em dias
base['tempo_entrega_real'] = (base['order_delivered_customer_date'] - base['order_purchase_timestamp']).dt.days
# preencher com nulo linhas menores que zero.
base['tempo_entrega_real'] = np.where(base['tempo_entrega_real'] < 0, np.nan, base['tempo_entrega_real'])

# flag de atraso na entrega, indicando se o pedido chegou após a data estimada (1 = Atrasado, 0 = No prazo)
base['atraso_entrega'] = np.where(base['order_delivered_customer_date'] > base['order_estimated_delivery_date'], 1, 0)

# JANELA DE POSTAGEM DO VENDEDOR
base['janela_postagem'] = (base['shipping_limit_date'] - base['order_purchase_timestamp']).dt.days
base['janela_postagem'] = np.where(base['janela_postagem'] < 0, np.nan, base['janela_postagem'])

# Apenas ano e mês para análise de série temporal
base['mes_ano'] = base['order_purchase_timestamp'].dt.to_period('M')

# Tempo prometido ao cliente 
base['tempo_entrega_estimado'] = (base['order_estimated_delivery_date'] - base['order_purchase_timestamp']).dt.days
base['tempo_entrega_estimado'] = np.where(base['tempo_entrega_estimado'] < 0, np.nan, base['tempo_entrega_estimado'])

# QUANTIDADE DE ITENS
base['qtd_itens_pedido'] = base.groupby('order_id')['order_id'].transform('count')

In [ ]:
print('================================================')
print(f'Estrutura do dataframe ao final da limpeza: \n{base.shape[0]} linhas | {base.shape[1]} colunas')
print('================================================')

## **Etapa III - Análise exploratória dos dados (EDA)**

#### Realiza-se a exploração dos dados em uma cópia separada para entender o comportamento do negócio. Em seguida, na etapa de modelagem, será separada previamente os conjuntos de treino e teste antes de qualquer transformação estatística, evitando vazamento de dados e garantindo que o modelo tenha performance real em dados novos.

In [ ]:
# cópia de segurança e para exploração
df = base.copy()

### **Análise macro vendas / faturamento**

In [ ]:
# 1. Faturamento Total Geral
faturamento_total = df['valor_total'].sum()
qtd_pedidos = df['order_id'].nunique()
ticket_medio = faturamento_total / qtd_pedidos

print(f"📊 RESULTADO DA ANÁLISE:")
print(f"Faturamento Total Período: R$ {faturamento_total:,.2f}".replace(',', '.'))
print(f"Quantidade de Pedidos: {qtd_pedidos}")
print(f"Ticket Médio Geral: R$ {ticket_medio:.2f}")

### **Análise vendas por mês**

In [ ]:
# Faturamento mensal
faturamento_mes = df.groupby('mes_ano')['valor_total'].sum().reset_index()

# transformando a coluna de datas em texto
faturamento_mes['mes_ano'] = faturamento_mes['mes_ano'].astype(str)

#configurando e plotando o gráfico de linha
plt.figure(figsize=(10,4))

sns.lineplot(
    data=faturamento_mes, 
    x='mes_ano',        # Nome exato da coluna X (agora como texto)
    y='valor_total',    # Nome exato da coluna Y
    marker='o'   
)

plt.title('Faturamento Mensal (R$)')
plt.xticks(rotation=45)
plt.xlabel('')
plt.tight_layout()
plt.show()

#### segundo pesquisas feitas sobre essa base, no primeiros meses de coleta, de setembro e dezembro de 2016, o sistema quase não registrou vendas, causando esse 'bug' de aparente queda nas vendas em pleno período de final do ano.
#### com excessão desses primeiros meses de coleta, analisando de forma geral, o faturamento segue estável e firme, alcançando ápice de mais de 1 milhão de faturamento em novembro de 2017, possível efeito de black friday, mais até que dezembro onde cai um pouco o faturamento, 
#### demonstrando sazonalidade. Até o fim da coleta, o faturamento alto mostra a maturidade da plataforma.

#### o ticket médio geral é de: R$ 160.01

### **Análise da satisfação dos clientes através da nota**

In [ ]:
# classificando as pontuações
df['avaliacao'] = np.where(df['review_score'] >= 4, 'ótima', 'nota menor')

df['avaliacao'].value_counts(normalize=True).plot(kind='bar')

plt.title("Proporção de notas boas vs ruim")
plt.ylabel("percentual")
plt.ylim(0, 1)
plt.xticks(rotation=0)

plt.show()

### **Análise da correlação linear com o alvo**

In [ ]:
# Selecionando apenas as colunas que são números
df_numerico = df.select_dtypes(include=['int', 'float'])

# calculando a matriz completa e filtrando apenas para a coluna do alvo
correlacoes_alvo = df_numerico.corr()['alvo'].sort_values(ascending=False).reset_index().round(2)

print("🔗 Correlação das variáveis com a Nota Ruim (Alvo):")
display(correlacoes_alvo.head(5))


#### a correlação moderada entre atraso nas entregas e pontuação da nota de 0.31 confirma o impacto na nota causado por pedidos atrasados.

#### é possível notar também que há uma correlação moderada entre o tempo de entrega e a nota de avaliação (0.29), ou seja,

#### quanto maior o tempo para entregar, menor a nota de avaliação.

### **Análise bivariada: Tempo de entrega Vs nota**

In [ ]:
# Média da nota por faixa de tempo de entrega
# Criando faixas para ver a evolução
df['faixa_entrega'] = pd.cut(
    x=df['tempo_entrega_real'],
    bins=[0, 5, 10, 15, 20, 90],
    labels=['0-5 dias', '6-10 dias', '11-15 dias', '16-20 dias', '> 20 dias']
)

media_nota_entrega = df.groupby('faixa_entrega', observed=True)['review_score'].agg(['mean', 'count']).round(2)
print("\n⭐ Média da Nota conforme tempo de entrega:")
print(media_nota_entrega)

# Comparando Atraso vs Nota
media_nota_atraso = df.groupby('atraso_entrega')['review_score'].mean().round(2)
print("\n⏱️ Média da Nota: Pedidos NO PRAZO vs ATRASADOS")
print(f"No Prazo: {media_nota_atraso[0]}")
print(f"Atrasado: {media_nota_atraso[1]}")

#### Nota-se que em média, pedidos entregues em até 5 dias tem nota de ~4.4, enquanto pedidos que demoram mais de 20 dias tem nota média de ~3.0

#### pedidos entregues com atraso tem uma média de nota ~40% inferior comparado aos pedidos entregues no prazo

#### essa análise indica um desalinhamento sobre a data prometida de entrega que compromete a expectativa do cliente.

### **Análise macro das melhores categorias**

In [ ]:
# agrupando por categoria
categorias=df.groupby('product_category_name')['valor_total'].sum()

categorias = pd.DataFrame(categorias)
categorias['pct_faturamento'] = (categorias['valor_total'] / categorias['valor_total'].sum())
display(categorias.reset_index().sort_values(by='valor_total', ascending=False).head())

# tabela visual das top categorias
sns.barplot(
    data=categorias.sort_values(by='valor_total', ascending=False).head(5),
    y='product_category_name',
    x='valor_total'
)

plt.title('Top 5 faturamento por categoria')
plt.xticks(rotation=45)
plt.ylabel('Categoria')
plt.xlabel('valor total')

plt.show()

#### categoria com maior faturamento geral aparece **'beleza/saúde'**, representando 9,2% do faturamento geral.

#### seguido de **'relógios/presentes'** e **'cama/mesa/banho'**. Essas três categorias juntas representam ~25% da receita.

### **análise macro por região**

In [ ]:
# 1. Desempenho por Estado
analise_estado = df.groupby('customer_state').agg(
    faturamento = ('valor_total', 'sum'),
    qtd_pedidos = ('order_id', 'nunique'),
    tempo_medio_entrega = ('tempo_entrega_real', 'mean'),
    nota_media = ('review_score', 'mean')
).reset_index().sort_values(by='faturamento', ascending=False)

analise_estado['pct_receita'] = (analise_estado['faturamento'] / faturamento_total).round(2)

print("🗺️ Desempenho por Estado:")
display(analise_estado.head().round(2))

# 2. Classificar Regiões para análise mais ampla
regioes = {
    'SP': 'Sudeste', 'RJ': 'Sudeste', 'MG': 'Sudeste', 'ES': 'Sudeste',
    'PR': 'Sul', 'SC': 'Sul', 'RS': 'Sul',
    'BA': 'Nordeste', 'PE': 'Nordeste', 'CE': 'Nordeste', 'RN': 'Nordeste', 'PB': 'Nordeste', 'AL': 'Nordeste', 'SE': 'Nordeste', 'MA': 'Nordeste', 'PI': 'Nordeste',
    'GO': 'Centro-Oeste', 'MT': 'Centro-Oeste', 'MS': 'Centro-Oeste', 'DF': 'Centro-Oeste',
    'AM': 'Norte', 'PA': 'Norte', 'AC': 'Norte', 'RO': 'Norte', 'RR': 'Norte', 'AP': 'Norte', 'TO': 'Norte'
}

df['regiao'] = df['customer_state'].map(regioes)

analise_regiao = df.groupby('regiao').agg(
    faturamento = ('valor_total', 'sum'),
    tempo_medio_entrega = ('tempo_entrega_real', 'mean'),
    qtd_pedidos = ('order_id', 'nunique'),
    nota_media = ('review_score', 'mean')
).reset_index()

analise_regiao['pct_receita'] = (analise_regiao['faturamento'] / faturamento_total).round(2)


print("\n🌐 Análise por GRANDE REGIÃO:")
display(analise_regiao.round(2).nlargest(5, columns='pct_receita'))

In [ ]:
fig, ax1 = plt.subplots(figsize=(10,4))

sns.barplot(
    data=analise_regiao.sort_values(by='faturamento', ascending=False),
    x='regiao',
    y='faturamento',
    ax=ax1,
    hue='regiao'
)
plt.xlabel('')

ax2 = ax1.twinx()

sns.lineplot(
    data=analise_regiao,
    x='regiao',
    y='tempo_medio_entrega',
    marker='o',
    color='red',
    ax=ax2
)
plt.title('faturamento por região e tempo médio de entrega')
plt.xlabel('')
plt.ylim(1, 30)
plt.show()

#### Analisando por região, sudeste concentra mais de 60% da receita, com menor tempo de entrega, média de 10 dias.

#### Já o norte sofre mais com o tempo de entrega, em média o dobro da região sudeste,

#### nordeste aparece em segundo lugar com maior tempo médio de entrega, indicando gargalos logísticos que podem ser oportunidade de crescimento.

### **Análise nota por região**

In [ ]:
# Gráfico de dispersão
sns.scatterplot(
    data=analise_regiao,
    y='nota_media',
    x='tempo_medio_entrega',
    hue='regiao',
    s=250
)

plt.title('Média da nota por Região')
plt.show()

#### Regiões norte e nordeste, além de apresentarem os maiores tempos para entregar, também possuem as menores média de notas ~3.9, confirmando gargalos logísticos que impactam na satisfação do cliente.

#### Sudeste e sul, em contrapartida, apresentam maiores notas e menor tempo de entrega, indicando bastante solidez operacional

### **Análise de pagamentos**

In [ ]:
#Distribuição dos tipos de pagamento
forma_pagamento = df['payment_type'].value_counts(normalize=True).mul(100).round(2)

print("💳 Participação % por Tipo de Pagamento:")
print(forma_pagamento)

# Valor médio do pedido por tipo de pagamento
valor_medio_pagamento = df.groupby('payment_type')['valor_total'].mean().round(2)
print("\n💰 Valor Médio do Pedido por Tipo de Pagamento:")
print(valor_medio_pagamento.sort_values(ascending=False))

sns.barplot(
    data=forma_pagamento
)

plt.title('Participação por tipo de pagamento')
plt.ylabel('percentual')
plt.xlabel('')
plt.xticks(rotation=0)

plt.show()

#### O cartão de crédito é o método dominante, utilizado em 73.8% dos pedidos, e apresenta o valor médio de compra mais elevado (R$ 145.83). 

#### O boleto ainda representa uma fatia consideravel (19.4%), sendo essencial para estratégias de venda para clientes que não possuem cartão.

## **Segmentação de clientes**

#### buscando interpretabilidade, nessa etapa não se optou em dividir a base por quartis para segmentar os clientes, e sim em regras de negócio claras, fixas e explicáveis baseada nos dados disponíveis:

- 💎 CLIENTE VIP:  se comprou mais de 1 vez ou gastou mais de R$ 1.000 em uma única compra, pois quem compra mais de uma vez é fiel e quem gasta muito, mesmo uma vez, é valioso.

- ⭐ CLIENTE POTENCIAL / BOM CLIENTE: se comprou nos últimos 6 meses e gastou entre R$ 300 e R$ 1.000, pois quem gasta um valor bom e comprou recentemente tem chance de voltar.

- 🔄 CLIENTE COMUM: se comprou nos últimos 6 meses e gastou menos de R$ 300, pois Compra pouco, mas ainda está "ativo".

- ⚠️ CLIENTE INATIVO / BAIXO VALOR: se comprou há mais de 6 meses ou gastou muito pouco e não voltou. pois provavelmente não volta mais ou tem baixo retorno.

In [ ]:
# Definindo uma data de referência (última data da base + 1 dia)
data_max = df['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

# Calculando as métricas RFM por cliente
rfm = df.groupby('customer_unique_id').agg(
    # Recência: Quantos dias desde a última compra
    ultima_compra = ('order_purchase_timestamp', 'max'),
    
    # Frequência: Quantas compras ele fez no total
    Frequencia_Compras = ('order_id', 'nunique'),
    
    # Valor: Quanto ele gastou no total
    Valor_Gasto_Total = ('valor_total', 'sum')
).reset_index()

# criando recencia: diferença entre a data de referência e a ultima compra
rfm['Recencia'] = (data_max - rfm['ultima_compra']).dt.days
rfm = rfm.drop('ultima_compra', axis=1)

rfm.head()

In [ ]:
# REGRAS DE NEGÓCIO

condicoes = [
        (rfm['Frequencia_Compras'] > 1) | (rfm['Valor_Gasto_Total'] > 1000),
        (rfm['Recencia'] <= 180) & (rfm['Valor_Gasto_Total'] >= 300),
        (rfm['Recencia'] <= 180)
]

rotulos = ['💎 VIP', '⭐ Potencial', '🔄 Comum']
    
rfm['segmento'] = np.select(condicoes, rotulos, default= '⚠️ Inativo')


# 4. 📊 Resultado para Análise
print("📊 DISTRIBUIÇÃO DOS CLIENTES POR PERFIL:\n")
display(rfm['segmento'].value_counts())

print("\n📊 PERCENTUAL DOS CLIENTES POR PERFIL:\n")
display((rfm['segmento'].value_counts(normalize=True)*100).round(2))


# 5. 📈 Resumo estatístico por segmento
resumo_segmento = rfm.groupby('segmento').agg(
    qtd_clientes = ('customer_unique_id', 'nunique'),
    recencia_media = ('Recencia', 'mean'),
    frequencia_media = ('Frequencia_Compras', 'mean'),
    valor_medio_gasto = ('Valor_Gasto_Total', 'mean'),
    valor_total_gasto = ('Valor_Gasto_Total', 'sum')
).round(2)

resumo_segmento['pct_clientes'] = (resumo_segmento['qtd_clientes'] / resumo_segmento['qtd_clientes'].sum()).round(2)
resumo_segmento['pct_receita'] = (resumo_segmento['valor_total_gasto'] / resumo_segmento['valor_total_gasto'].sum()).round(2)


print("\n📈 RESUMO DAS CARACTERÍSTICAS POR SEGMENTO:\n")
display(resumo_segmento)

In [ ]:
fig, ax1 = plt.subplots(figsize=(8,4))

sns.barplot(
    data=resumo_segmento,
    x='segmento',
    y='pct_clientes',
    ax=ax1
)

plt.ylabel('percentual')
plt.xlabel('')
plt.ylim(0.0, 1.0)

ax2 = ax1.twinx()

ax2 = sns.lineplot(
    data=resumo_segmento,
    x='segmento',
    y='valor_medio_gasto',
    ax=ax2,
    marker='o',
    color='red'
)

plt.title('Proporção de clientes por segmento')
plt.ylabel('valor médio gasto')
plt.ylim(100, 800)
plt.xticks([0, 1, 2 , 3], ['Inativo', 'Potencial', 'VIP', 'Comum'])
plt.xlabel('')
plt.show()

#### a base de clientes apresenta uma característica marcante do e-commerce: baixa recorrência de compra, mais de 95% dos clientes compram apenas uma vez. 

#### Devido a isso, a segmentação foi feita por regras de negócio, priorizando o valor gasto e a data da última compra. Apenas ~4% dos clientes são considerados VIPs (alto valor ou recorrentes), enquanto quase metade da base já está inativa há mais de 6 meses. 

#### Clientes vips e potenciais estão no topo da 'pirâmide' de gastos médios, mostrando sua força de impacto na receita.

#### analisando a segmentação realizada, nota-se que a maior oportunidade de crescimento não é apenas atrair novos clientes, mas criar campanhas para reativar ou fidelizar os clientes que já compraram.

## **Consolidando a base para a etapa de modelagem**

In [ ]:
# desconsiderando pedidos sem notas de avaliação (-1)
df = df[df['review_score'] != -1].copy()

# criando a variável alvo
df['avaliacao_ruim'] = np.where(df['review_score'] < 4, 1, 0)

# excluindo colunas que não serão usadas nas proximas etapas
df = df.drop(columns=[
    'customer_id', 'product_id', # excluindo IDs
    'customer_city',  # texto bruto que será transformado para evitar muita granulidade
    'order_status', # coluna removida pois pedidos da base estão entregues e isso evita vazamento.
    'avaliacao', 'alvo',               # deriva do alvo 
    'faixa_entrega',                 # já transformada
    'seller_city',   # não usada no modelo
    'nao_avaliado' # não usada no modelo
])

print(f"Base tratada: {df.shape[0]} linhas | {df.shape[1]} colunas")

In [ ]:
#salvando base tratada
df.to_pickle('../bases/base_tratada/dados_limpos.pkl')

## **Conclusões iniciais - visão macro**

#### As notas possuem uma distribuição concentrada e menos dispersão, o que dificultou a segmentação simples por perfil de cliente ou características básicas do pedido. 

#### Este comportamento não reflete uma homogeneidade real na experiência do cliente, mas sim um efeito matemático provocado pelo forte desbalanceamento da base. Como aproximadamente 77% das avaliações históricas do ecossistema são notas máximas (4 e 5), o cálculo da média simples acaba mascarando os 23% de detratores (notas 1, 2 e 3). 

#### na Correlação tempo de entrega vs satisfação do cliente: a média da nota entre pedidos entregues no prazo e atrasados, chega a cair de 4.18 para 2.47, porém é informação futura que não entra no modelo.

#### Com isso, há uma necessidade de algoritmos baseados em Árvores que lidam melhor com a concentração das médias e conseguem fatiar os dados em regras específicas de comportamento para encontrar a classe rara.

________________________________________________________

#### - Faturamento mostra pico em datas comemorativas, mas em crescimento até novembro de 2017, depois leve queda, mas com alta média de faturamento nos meses após a leve queda.

#### Cartão de crédito é a majoritária forma de pagamento = 75% das vendas.

________________________________________________________________

#### entre as categorias mais rentáveis, estão:

**- beleza / saúde.**

**- relógios / presentes.**

**- cama/mesa/banho.**

_____________________________________________________________

#### Estados: **SP, RJ e MG** concentram mais de 60% do faturamento, já **Norte e Nordeste** sofrem mais com entrega lenta e notas mais baixas.

_____________________________________________________________________

#### Apenas ~8% dos clientes são **vips ou potenciais**, porém geram ~30% da receita total

## **Recomendações iniciais**

- Investir em logística para regiões Norte/Nordeste em busca de melhorar avaliação, repetição de compra e menores prazos de entregas.

- Focar em campanhas de fidelidade para clientes VIP e potenciais.

- priorizar aumento de estoque das categorias de maior faturamento.

- melhorar previsão de entrega com aviso prévio, alinhando expectativas e reduzindo reclamações e notas baixas.